# Lab 02.1 — DataFrame e Spark SQL

## Introduzione
Un **DataFrame** è una collezione distribuita di righe con uno **schema** (colonne con nome e tipo), come una tabella.
Spark offre due linguaggi equivalenti per interrogarlo: la **API DataFrame** (Python, Scala, Java, R) e **SQL** standard (ANSI), tramite *viste temporanee* o tabelle.
Entrambi producono lo stesso **piano logico**, ottimizzato dal motore **Catalyst** ed eseguito da **Tungsten** (codice generato a runtime, memoria off-heap).

Il dataset è un e-commerce sintetico (clienti, prodotti, ordini) generato in parallelo dai task Spark, che in modalità `local[*]` sono thread del driver (vedi `dati_shop.py`).

### Obiettivi
1. Inizializzare la `SparkSession` e caricare le tabelle dell'e-commerce, ispezionandone lo schema.
2. Interrogare i dati con la **API DataFrame** e con **SQL**, verificando che producono lo stesso piano.
3. Eseguire join tra più tabelle e aggregazioni multidimensionali (`ROLLUP`, `pivot`).
4. Usare le **funzioni finestra** (ranking, totali cumulati, variazioni).
5. Confrontare funzioni built-in, **UDF Python** e **Pandas UDF**.
6. Leggere il piano di **Catalyst** (predicate pushdown, column pruning, broadcast join).
7. Chiudere la sessione con `spark.stop()`.

## Step 1: Inizializzare la SparkSession

La `SparkSession` è il punto di ingresso per DataFrame e SQL. Il master arriva dalla variabile d'ambiente `SPARK_MASTER` (nel lab 02 è `local[*]`: gli executor sono thread dentro il driver, quindi non serve alcun cluster).
Impostiamo `spark.sql.shuffle.partitions` a 8: il default (200) produrrebbe troppi task minuscoli su dati di queste dimensioni.

In [ ]:
# Importa PySpark, le funzioni SQL (F), le finestre (Window) e il generatore di dati del lab
import os, time
from pyspark.sql import SparkSession, functions as F, Window
import dati_shop

spark = (SparkSession.builder
         .appName("Lab02-DataFrame-SQL")
         .master(os.environ.get("SPARK_MASTER", "local[*]"))
         .config("spark.sql.shuffle.partitions", "8")   # default 200: troppe per dati piccoli
         .getOrCreate())
print("Spark", spark.version, "| master:", spark.sparkContext.master)

## Step 2: Creare i DataFrame e ispezionare lo schema

`dati_shop.carica()` genera in parallelo 2 milioni di ordini (più clienti e prodotti) e li salva in Parquet in `data/shop/`: dalle esecuzioni successive li rilegge soltanto.
`printSchema()` mostra nomi e tipi delle colonne, `show(5)` le prime righe, `count()` è la prima **azione** (fa partire il calcolo).

In [ ]:
# Al primo avvio genera i dati in parallelo e li salva in Parquet (data/shop/)
tabelle  = dati_shop.carica(spark, n_ordini=2_000_000)
clienti, prodotti, ordini = tabelle["clienti"], tabelle["prodotti"], tabelle["ordini"]

ordini.printSchema()
ordini.show(5)
print(f"ordini: {ordini.count():,}  clienti: {clienti.count():,}  prodotti: {prodotti.count()}")

### Step 2b: Statistiche descrittive

`summary()` calcola in parallelo conteggio, minimo, massimo e media delle colonne scelte: un modo rapido per capire la forma dei dati prima di interrogarli.

In [ ]:
# Statistiche descrittive calcolate in parallelo
ordini.select("quantita", "data").summary("count", "min", "max", "mean").show()

## Step 3: Interrogare con la API DataFrame

Le **trasformazioni** (`select`, `where`, `withColumn`, `groupBy`, `join`...) sono *lazy*: costruiscono solo un piano. Le **azioni** (`show`, `count`, `collect`, `write`...) lo eseguono.
Qui uniamo ordini e prodotti, calcoliamo `importo` e `mese`, poi aggreghiamo il fatturato 2024 per categoria.

In [ ]:
righe = (ordini
         .join(prodotti, "prodotto_id")                                   # join sulla chiave comune
         .withColumn("importo", F.col("quantita") * F.col("prezzo"))      # colonna calcolata
         .withColumn("mese", F.date_trunc("month", "data").cast("date")))

fatturato_categoria = (righe
    .where(F.year("data") == 2024)
    .groupBy("categoria")
    .agg(F.count("*").alias("ordini"),
         F.sum("importo").alias("fatturato"),
         F.round(F.avg("quantita"), 2).alias("q_media"))
    .orderBy(F.desc("fatturato")))

fatturato_categoria.show()

## Step 4: La stessa interrogazione in SQL

Registriamo i DataFrame come **viste temporanee**: diventano tabelle interrogabili con `spark.sql(...)` nella sessione corrente (senza copiare i dati).
La query seguente è equivalente a quella dello Step 3 e restituisce un DataFrame.

In [ ]:
clienti.createOrReplaceTempView("clienti")
prodotti.createOrReplaceTempView("prodotti")
ordini.createOrReplaceTempView("ordini")

sql_df = spark.sql("""
    SELECT p.categoria,
           COUNT(*)                         AS ordini,
           SUM(o.quantita * p.prezzo)       AS fatturato,
           ROUND(AVG(o.quantita), 2)        AS q_media
    FROM ordini o
    JOIN prodotti p USING (prodotto_id)
    WHERE year(o.data) = 2024
    GROUP BY p.categoria
    ORDER BY fatturato DESC
""")
sql_df.show()

### Step 4b: SQL e DataFrame producono lo stesso piano

`explain(mode="formatted")` mostra il piano fisico scelto da Catalyst. SQL e API DataFrame sono solo due *front-end* diversi: confrontando questo piano con quello della versione DataFrame si trovano le stesse operazioni.

In [ ]:
# SQL e DataFrame sono solo due "front-end": Catalyst produce lo stesso piano fisico
sql_df.explain(mode="formatted")

### Punti chiave
* DataFrame e SQL sono equivalenti: scegli quello più comodo, le prestazioni sono le stesse.
* Le trasformazioni sono *lazy*; solo le azioni avviano il calcolo.
* Una vista temporanea non copia i dati: dà un nome a un piano di calcolo.

## Step 5: Join tra più tabelle

Uniamo ordini, clienti e prodotti per calcolare ordini e fatturato per città e canale di vendita. Le condizioni di join sono scritte con `ON` perché le chiavi hanno nomi diversi tra le tabelle.

In [ ]:
spark.sql("""
    SELECT c.citta, o.canale, COUNT(*) AS ordini, ROUND(SUM(o.quantita * p.prezzo), 0) AS fatturato
    FROM ordini o
    JOIN clienti  c ON o.cliente_id  = c.cliente_id
    JOIN prodotti p ON o.prodotto_id = p.prodotto_id
    GROUP BY c.citta, o.canale
    ORDER BY c.citta, o.canale
""").show(12)

### Step 5b: Subtotali con ROLLUP

`GROUP BY ROLLUP(a, b)` produce i gruppi `(a, b)`, i subtotali per `a` e il totale generale: le righe di subtotale hanno `NULL` nelle colonne aggregate. `NULLS LAST` le mette in fondo a ciascun gruppo.

In [ ]:
# ROLLUP: subtotali per città e totale generale (righe con NULL)
spark.sql("""
    SELECT c.citta, o.canale, COUNT(*) AS ordini
    FROM ordini o JOIN clienti c USING (cliente_id)
    WHERE c.citta IN ('Roma', 'Milano')
    GROUP BY ROLLUP(c.citta, o.canale)
    ORDER BY c.citta NULLS LAST, o.canale NULLS LAST
""").show()

### Step 5c: Tabella pivot

`pivot` trasforma i valori di una colonna (qui l'anno) in colonne. Passare esplicitamente l'elenco dei valori `[2024, 2025]` evita a Spark un passaggio in più per scoprirli.

In [ ]:
# Tabella pivot: fatturato per categoria (righe) e anno (colonne)
(righe.withColumn("anno", F.year("data"))
      .groupBy("categoria").pivot("anno", [2024, 2025]).agg(F.round(F.sum("importo"), 0))
      .orderBy("categoria").show())

## Step 6: Funzioni finestra — ranking per categoria

Le **window function** calcolano un valore per ogni riga guardando un insieme di righe "vicine" (la finestra), **senza** collassare le righe come fa `GROUP BY`.
Qui `RANK() OVER (PARTITION BY categoria ORDER BY fatturato DESC)` assegna una posizione a ogni prodotto dentro la propria categoria; il filtro esterno tiene i primi 3.

In [ ]:
# Top 3 prodotti per fatturato in ogni categoria
spark.sql("""
    WITH vendite AS (
        SELECT p.categoria, p.prodotto, SUM(o.quantita * p.prezzo) AS fatturato
        FROM ordini o JOIN prodotti p USING (prodotto_id)
        GROUP BY p.categoria, p.prodotto
    )
    SELECT * FROM (
        SELECT *, RANK() OVER (PARTITION BY categoria ORDER BY fatturato DESC) AS posizione
        FROM vendite
    ) WHERE posizione <= 3
    ORDER BY categoria, posizione
""").show(12)

### Step 6b: Totale cumulato e variazione mensile

Con l'API DataFrame definiamo una `Window` ordinata per mese: `sum().over(rowsBetween(...))` dà il cumulato, `lag()` il valore del mese precedente da cui ricavare la variazione percentuale.
La finestra non ha `PARTITION BY`: Spark avvisa che userà una sola partizione, ma i dati aggregati sono pochi e va bene.

In [ ]:
# Andamento mensile: totale cumulato e variazione rispetto al mese precedente (API DataFrame)
# (finestra senza PARTITION BY: pochi dati aggregati, Spark avvisa che userà una sola partizione)
mensile = righe.groupBy("mese").agg(F.round(F.sum("importo"), 0).alias("fatturato"))
w = Window.orderBy("mese")
(mensile
   .withColumn("cumulato", F.sum("fatturato").over(w.rowsBetween(Window.unboundedPreceding, 0)))
   .withColumn("mese_prec", F.lag("fatturato").over(w))
   .withColumn("variazione_%", F.round((F.col("fatturato") / F.col("mese_prec") - 1) * 100, 1))
   .orderBy("mese").show(8))

### Punti chiave
* `ROLLUP` e `pivot` producono report multidimensionali con una sola query.
* Le window function aggiungono informazioni a ogni riga (rank, cumulato, valore precedente) senza ridurre il numero di righe.
* `PARTITION BY` nella finestra permette a Spark di calcolare le partizioni in parallelo.

## Step 7: Funzioni definite dall'utente (UDF)

Le funzioni **built-in** girano nella JVM e sono ottimizzate da Catalyst. Una **UDF Python** richiede di serializzare ogni riga verso un processo Python: è più lenta. Le **Pandas UDF** (vettoriali) trasferiscono blocchi di righe in formato **Apache Arrow** e sono molto più efficienti.
Applichiamo la stessa operazione (IVA al 22%) nei tre modi e ne misuriamo i tempi.

In [ ]:
import pandas as pd
from pyspark.sql.functions import udf, pandas_udf

@udf("double")
def iva_udf(x):
    return round(float(x) * 1.22, 2)

@pandas_udf("double")
def iva_pandas(x: pd.Series) -> pd.Series:
    return (x.astype("float64") * 1.22).round(2)

base = righe.select(F.col("importo").cast("double").alias("importo"))

for nome, espr in [("built-in", F.col("importo") * 1.22),
                   ("UDF Python", iva_udf("importo")),
                   ("Pandas UDF", iva_pandas("importo"))]:
    t0 = time.time()
    base.select(F.sum(espr)).collect()
    print(f"{nome:12s}: {time.time() - t0:.2f}s")

### Step 7b: Usare una UDF in SQL

`spark.udf.register` rende la funzione disponibile con un nome anche nelle query SQL.

In [ ]:
# Le UDF si possono registrare e usare anche in SQL
spark.udf.register("iva", iva_pandas)
spark.sql("SELECT prodotto, prezzo, iva(prezzo) AS prezzo_ivato FROM prodotti LIMIT 5").show()

### Punti chiave
* Preferire sempre le funzioni built-in: Catalyst può ottimizzarle.
* Se serve logica Python personalizzata, una Pandas UDF è molto più veloce di una UDF a riga singola.

## Step 8: Catalyst — dal SQL al piano fisico

`explain(extended=True)` mostra le fasi dell'ottimizzatore: piano **logico analizzato** → **ottimizzato** → **fisico**.
Nella query seguente osserviamo come il filtro sulla città viene spostato **prima** del join (*predicate pushdown*) e come vengono lette solo le colonne necessarie (*column pruning*).

In [ ]:
q = spark.sql("""
    SELECT c.nome, o.data
    FROM ordini o JOIN clienti c USING (cliente_id)
    WHERE c.citta = 'Bari' AND o.quantita > 4
""")
q.explain(extended=True)

Nel piano fisico cerca:
* `PushedFilters: [IsNotNull(citta), EqualTo(citta,Bari)]` → il filtro è passato al lettore Parquet;
* `ReadSchema` → vengono lette solo le colonne usate;
* `BroadcastHashJoin` → la tabella clienti (piccola) è copiata in memoria per ogni task invece di fare uno shuffle.

### Punti chiave
* Catalyst riscrive il piano: filtri spinti verso la sorgente, colonne inutili scartate, strategia di join scelta in base alla dimensione.
* `explain()` è lo strumento principale per capire e ottimizzare una query.

## Step 9: Chiudere la SparkSession

`spark.stop()` rilascia le risorse (e libera la Spark UI sulla porta 4040).

In [ ]:
spark.stop()

## Riepilogo

| Concetto | API / comando | A cosa serve |
|---|---|---|
| Sessione | `SparkSession.builder...getOrCreate()` | punto di ingresso per DataFrame e SQL |
| Schema e anteprima | `printSchema()`, `show()`, `summary()` | ispezionare struttura e statistiche |
| API DataFrame | `join`, `withColumn`, `groupBy().agg()` | trasformare e aggregare in modo programmatico |
| Viste temporanee | `createOrReplaceTempView`, `spark.sql()` | interrogare i DataFrame con SQL |
| Aggregazioni multidimensionali | `ROLLUP`, `pivot` | subtotali e tabelle pivot |
| Funzioni finestra | `RANK() OVER`, `sum().over()`, `lag()` | ranking, cumulati, confronto con la riga precedente |
| UDF | `@udf`, `@pandas_udf`, `spark.udf.register` | logica Python personalizzata (meglio vettoriale) |
| Catalyst | `explain(extended=True)` | leggere pushdown, pruning e strategia di join |
| Chiusura | `spark.stop()` | rilasciare le risorse |